In [78]:
!pip install nltk
import numpy as np
import pandas as pd
import re
import emoji
import nltk
import tensorflow as tf

from nltk.corpus import sentiwordnet as swn
from nltk.corpus import wordnet as wn
from nltk import pos_tag, word_tokenize

from sklearn.preprocessing import StandardScaler

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Input, Embedding, Conv1D, GlobalMaxPooling1D, Dense, Concatenate, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping



In [79]:
print("GPU:", tf.config.list_physical_devices('GPU'))


GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [80]:
train = pd.read_csv("/kaggle/input/csi-ml-sprint/train.csv")

# remove hidden spaces from column names
train.columns = train.columns.str.strip()

print("Shape:", train.shape)
print("\nColumns:", train.columns.tolist())

train.head()


Shape: (9961, 3)

Columns: ['id', 'text', 'label']


,id,text,label
0,1,i didnt feel humiliated 😔,0
1,2,i can go from feeling so hopeless 😞 to so damn...,0
2,3,im grabbing a minute to post i feel greedy 🤑 w...,3
3,4,i am ever feeling nostalgic 🕰️ about the firep...,2
4,5,i am feeling grouchy 😠,3


In [81]:
def clean_text(text):
    text = text.lower()
    text = emoji.demojize(text)          # 😢 → :crying_face:
    text = re.sub(r"http\S+", "", text)  # remove URLs
    text = re.sub(r"@\w+", "", text)     # remove mentions
    text = re.sub(r"#", "", text)        # keep hashtag words
    text = re.sub(r"\s+", " ", text).strip()
    return text

train["text"] = train["text"].apply(clean_text)

train["text"].head()


0               i didnt feel humiliated :pensive_face:
1    i can go from feeling so hopeless :disappointe...
2    im grabbing a minute to post i feel greedy :mo...
3    i am ever feeling nostalgic :mantelpiece_clock...
4                    i am feeling grouchy :angry_face:
Name: text, dtype: object

In [82]:
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('wordnet')
nltk.download('sentiwordnet')


[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /usr/share/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package sentiwordnet to
[nltk_data]     /usr/share/nltk_data...
[nltk_data]   Package sentiwordnet is already up-to-date!


True

In [83]:
def get_sentiment_scores(text):
    tokens = word_tokenize(text)
    tagged = pos_tag(tokens)

    pos_score = 0
    neg_score = 0
    obj_score = 0
    count = 0

    for word, tag in tagged:
        wn_tag = None

        if tag.startswith('J'):
            wn_tag = wn.ADJ
        elif tag.startswith('V'):
            wn_tag = wn.VERB
        elif tag.startswith('N'):
            wn_tag = wn.NOUN
        elif tag.startswith('R'):
            wn_tag = wn.ADV

        if wn_tag:
            synsets = wn.synsets(word, pos=wn_tag)
            if synsets:
                swn_syn = swn.senti_synset(synsets[0].name())
                pos_score += swn_syn.pos_score()
                neg_score += swn_syn.neg_score()
                obj_score += swn_syn.obj_score()
                count += 1

    if count == 0:
        return [0, 0, 0]

    return [pos_score/count, neg_score/count, obj_score/count]


In [84]:
sent_features = np.array([get_sentiment_scores(t) for t in train["text"]])

sent_features[:5]


array([[0.04166667, 0.20833333, 0.75      ],
       [0.1015625 , 0.1328125 , 0.765625  ],
       [0.04166667, 0.16666667, 0.79166667],
       [0.05681818, 0.05681818, 0.88636364],
       [0.125     , 0.125     , 0.75      ]])

In [85]:
scaler = StandardScaler()
sent_scaled = scaler.fit_transform(sent_features)

sent_scaled[:5]


array([[-0.95477088,  1.74390037, -0.68530189],
       [ 0.1493572 ,  0.51161148, -0.49090128],
       [-0.95477088,  1.06401684, -0.16690027],
       [-0.67546575, -0.72840336,  1.01128522],
       [ 0.58140732,  0.38413332, -0.68530189]])

In [86]:
tokenizer = Tokenizer(num_words=20000, oov_token="<OOV>")
tokenizer.fit_on_texts(train["text"])

sequences = tokenizer.texts_to_sequences(train["text"])

padded_seq = pad_sequences(
    sequences,
    maxlen=50,          # optimal for tweets
    padding='post',
    truncating='post'
)

padded_seq.shape


(9961, 50)

In [87]:
import tensorflow as tf

def focal_loss(gamma=2., alpha=0.25):
    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        ce = tf.keras.losses.categorical_crossentropy(y_true, y_pred)
        pt = tf.reduce_sum(y_true * y_pred, axis=-1)
        return alpha * tf.pow(1 - pt, gamma) * ce
    return loss


In [88]:
# ===== TextCNN branch =====
text_input = Input(shape=(50,))

embedding = Embedding(20000, 150)(text_input)   # slightly larger embedding improves semantics

conv3 = Conv1D(160, 3, activation='relu')(embedding)
conv4 = Conv1D(160, 4, activation='relu')(embedding)
conv5 = Conv1D(160, 5, activation='relu')(embedding)

pool3 = GlobalMaxPooling1D()(conv3)
pool4 = GlobalMaxPooling1D()(conv4)
pool5 = GlobalMaxPooling1D()(conv5)

cnn_features = Concatenate()([pool3, pool4, pool5])

# ===== Sentiment branch =====
sent_input = Input(shape=(sent_scaled.shape[1],))

# ===== Merge =====
merged = Concatenate()([cnn_features, sent_input])

dense = Dense(64, activation='relu')(merged)
drop = Dropout(0.3)(dense)

output = Dense(6, activation='softmax')(drop)

model = Model([text_input, sent_input], output)

# label smoothing improves generalization
loss_fn = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05)


model.compile(
    optimizer='adam',
    loss=loss_fn,
    metrics=['accuracy']
)

model.summary()


Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_10      │ (None, 50)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_5         │ (None, 50, 150)   │  3,000,000 │ input_layer_10[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_15 (Conv1D)  │ (None, 48, 160)   │     72,160 │ embedding_5[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_16 (Conv1D)  │ (None, 47, 160)   │     96,160 │ embedding_5[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_17 (Conv1D)  │ (None, 46, 160)   │    120,160 │ embedding_5[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 160)       │          0 │ conv1d_15[0][0]   │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 160)       │          0 │ conv1d_16[0][0]   │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 160)       │          0 │ conv1d_17[0][0]   │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_10      │ (None, 480)       │          0 │ global_max_pooli… │
│ (Concatenate)       │                   │            │ global_max_pooli… │
│                     │                   │            │ global_max_pooli… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_11      │ (None, 3)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_11      │ (None, 483)       │          0 │ concatenate_10[0… │
│ (Concatenate)       │                   │            │ input_layer_11[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 64)        │     30,976 │ concatenate_11[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 64)        │          0 │ dense_10[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_11 (Dense)    │ (None, 6)         │        390 │ dropout_5[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,319,846 (12.66 MB)

 Trainable params: 3,319,846 (12.66 MB)

 Non-trainable params: 0 (0.00 B)

In [89]:
from tensorflow.keras.utils import to_categorical

y_train_cat = to_categorical(train["label"], num_classes=6)

y_train_cat[:3]


array([[1., 0., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0.]])

In [90]:
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3,
    patience=1,
    min_lr=1e-5
)


In [91]:
history = model.fit(
    [padded_seq, sent_scaled],
    y_train_cat,
    epochs=12,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early]
)


Epoch 1/12
281/281 ━━━━━━━━━━━━━━━━━━━━ 7s 15ms/step - accuracy: 0.5035 - loss: 1.3488 - val_accuracy: 0.9729 - val_loss: 0.3409
Epoch 2/12
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9470 - loss: 0.4058 - val_accuracy: 0.9880 - val_loss: 0.2961
Epoch 3/12
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9825 - loss: 0.3228 - val_accuracy: 0.9829 - val_loss: 0.2924
Epoch 4/12
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9949 - loss: 0.2929 - val_accuracy: 0.9870 - val_loss: 0.2830
Epoch 5/12
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9978 - loss: 0.2808 - val_accuracy: 0.9900 - val_loss: 0.2792
Epoch 6/12
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9989 - loss: 0.2742 - val_accuracy: 0.9900 - val_loss: 0.2740
Epoch 7/12
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9991 - loss: 0.2718 - val_accuracy: 0.9910 - val_loss: 0.2699
Epoch 8/12
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9995 - loss: 0.2680 - val_accuracy: 0

In [92]:
train_probs = model.predict([padded_seq, sent_scaled])
train_preds = np.argmax(train_probs, axis=1)

from sklearn.metrics import accuracy_score

train_acc = accuracy_score(train["label"], train_preds)
print("Training Accuracy:", train_acc)



312/312 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Training Accuracy: 0.9988956932034936


In [93]:
from sklearn.model_selection import train_test_split

X_seq_tr, X_seq_val, X_sent_tr, X_sent_val, y_tr, y_val = train_test_split(
    padded_seq,
    sent_scaled,
    train["label"],
    test_size=0.1,
    stratify=train["label"],
    random_state=42
)
val_probs = model.predict([X_seq_val, X_sent_val])
val_preds = np.argmax(val_probs, axis=1)

val_acc = accuracy_score(y_val, val_preds)
print("Validation Accuracy:", val_acc)


32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
Validation Accuracy: 1.0


In [94]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

cv_scores = []

for train_idx, val_idx in skf.split(padded_seq, train["label"]):

    X_seq_tr = padded_seq[train_idx]
    X_seq_v = padded_seq[val_idx]

    X_sent_tr = sent_scaled[train_idx]
    X_sent_v = sent_scaled[val_idx]

    y_tr = train["label"].iloc[train_idx]
    y_v = train["label"].iloc[val_idx]

    temp_model = tf.keras.models.clone_model(model)
    temp_model.compile(
        optimizer='adam',
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
        metrics=['accuracy']
    )

    y_tr_cat = tf.keras.utils.to_categorical(y_tr, 6)

    temp_model.fit(
        [X_seq_tr, X_sent_tr],
        y_tr_cat,
        epochs=3,
        batch_size=32,
        verbose=0
    )

    preds = temp_model.predict([X_seq_v, X_sent_v])
    preds = preds.argmax(axis=1)

    cv_scores.append(accuracy_score(y_v, preds))

print("Cross-Validation Accuracy:", sum(cv_scores)/len(cv_scores))


104/104 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
104/104 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
104/104 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
Cross-Validation Accuracy: 0.9480972368849079


In [95]:
test = pd.read_csv("/kaggle/input/csi-ml-sprint/test.csv")
test.columns = test.columns.str.strip()

print(test.shape)
test.head()


(2000, 2)


,id,text
0,1,💧 im feeling rather rotten so im not very ambi...
1,2,💔 im updating my blog because i feel shitty 😢
2,3,i never make her separate from me because i do...
3,4,🌻 🥳 ✨ 😄 i left with my bouquet of red and yell...
4,5,i was feeling a 💧 💀 🥺 little vain when i did t...


In [96]:
test["text"] = test["text"].apply(clean_text)

test["text"].head()


0    :droplet: im feeling rather rotten so im not v...
1    :broken_heart: im updating my blog because i f...
2    i never make her separate from me because i do...
3    :sunflower: :partying_face: :sparkles: :grinni...
4    i was feeling a :droplet: :skull: :pleading_fa...
Name: text, dtype: object

In [97]:
test_sent_features = np.array([get_sentiment_scores(t) for t in test["text"]])

test_sent_features[:5]


array([[0.15277778, 0.19444444, 0.65277778],
       [0.125     , 0.05      , 0.825     ],
       [0.03125   , 0.07291667, 0.89583333],
       [0.06666667, 0.025     , 0.90833333],
       [0.04166667, 0.08333333, 0.875     ]])

In [98]:
test_sent_scaled = scaler.transform(test_sent_features)

test_sent_scaled[:5]


array([[ 1.09346673,  1.51727253, -1.89490566],
       [ 0.58140732, -0.83965703,  0.24782102],
       [-1.14679315, -0.46572109,  1.12910377],
       [-0.49391742, -1.24758714,  1.28462425],
       [-0.95477088, -0.29575021,  0.86990296]])

In [99]:
test_sequences = tokenizer.texts_to_sequences(test["text"])

test_padded = pad_sequences(
    test_sequences,
    maxlen=60,
    padding='post',
    truncating='post'
)

test_padded.shape


(2000, 60)

In [100]:
test_probs = model.predict([test_padded, test_sent_scaled])

test_probs[:3]


63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step


array([[0.97124195, 0.00535257, 0.00767625, 0.0057096 , 0.00332546,
        0.0066941 ],
       [0.97602385, 0.00348743, 0.00607868, 0.0031965 , 0.0058071 ,
        0.00540654],
       [0.94126475, 0.01252161, 0.0154114 , 0.00990408, 0.00910568,
        0.01179251]], dtype=float32)

In [101]:
final_preds = np.argmax(test_probs, axis=1)

final_preds[:10]


array([0, 0, 0, 1, 0, 4, 3, 1, 1, 3])

In [102]:
submission = pd.DataFrame({
    "id": test["id"],
    "label": final_preds
})

submission.to_csv("submission_textcnn_final.csv", index=False)

submission.head()


,id,label
0,1,0
1,2,0
2,3,0
3,4,1
4,5,0
